In [0]:
%run "/Workspace/Users/imsuryya@gmail.com/databricks/source_to_bronze/utils"

In [0]:
def write_delta(df, path, mode="overwrite"):
    (df.write
       .format("delta")
       .mode(mode)
       .save(path))

def read_delta(path):
    return spark.read.format("delta").load(path)

In [0]:

df3 = read_delta("/silver/Employee_info/dim_employee")
df1 = read_delta("/Volumes/assingment/silver/country/")
df2 = read_delta("/Volumes/assingment/silver/department/")

df3.display()

In [0]:
from pyspark.sql.functions import sum, col

Q1 = (df3.groupBy("department")
         .agg(sum("salary").alias("total_salary"))
         .orderBy(col("total_salary").desc()))

Q1.display()

In [0]:
Q2 = (df3.groupBy(col("country"), col("department"))
         .count())

Q2.display()

In [0]:
from pyspark.sql.functions import collect_list

Q3 = (df3.groupBy("department")
         .agg(collect_list("country").alias("countries")))

Q3.display()

In [0]:
from pyspark.sql.functions import avg

Q4 = (df3.groupBy("department")
         .agg(avg("age").alias("avg_age_of_dept")))

Q4.display()

In [0]:
from pyspark.sql.functions import current_date

Q5 = df3.withColumn("at_load_date", current_date())
Q5.display()

In [0]:
# Write to gold layer: fact_employee table
from datetime import date

today = date.today().isoformat()

(Q5.write
   .format("delta")
   .partitionBy("at_load_date")
   .mode("overwrite")
   .option("replaceWhere", f"at_load_date = '{today}'")
   .save("/gold/employee/fact_employee"))